# Requirement Classification from an NVMe Specification — Results Summary

**EECE5644 — Final Project**  ·  Ajaya Nath Chittela

This notebook loads the artifacts produced by the pipeline
(`python -m scripts.run_pipeline`) and summarizes the results. It reads the
metrics in `outputs/metrics/` and the figures in `outputs/figures/`, so re-run
the pipeline first if you want to regenerate everything from scratch.

**Pipeline:** NVMe spec PDF → Docling parse → weak modality labels →
TF-IDF baselines (NB / LogReg / Linear SVM) + fine-tuned DistilBERT → evaluation.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
MET = ROOT / 'outputs' / 'metrics'
FIG = ROOT / 'outputs' / 'figures'
DATA = ROOT / 'data' / 'dataset'

def load(name):
    return json.loads((MET / name).read_text())

base = load('metrics_baseline_logreg.json')
dbert = load('metrics_distilbert.json')
base_mask = load('metrics_baseline_logreg_masked.json')
dbert_mask = load('metrics_distilbert_masked.json')
tune = load('tuning.json')
stats = json.loads((DATA / 'label_stats.json').read_text())
print('artifacts loaded from', MET)

## 1. Dataset & class distribution

Labels are weak supervision from the spec's own normative modal language. The
corpus is heavily imbalanced — a realistic property of requirement documents.

In [ ]:
order = ['MANDATORY', 'PROHIBITED', 'RECOMMENDED', 'OPTIONAL', 'INFORMATIVE']
total = sum(stats.values())
dist = pd.DataFrame({
    'count': [stats.get(k, 0) for k in order],
    'share': [f"{100*stats.get(k, 0)/total:.1f}%" for k in order],
}, index=order)
print('total labeled sentences:', total)
display(dist)
display(Image(str(FIG / 'eda' / 'class_distribution.png')))

## 2. Exploratory data analysis

No single surface feature strongly correlates with the label; the modal-cue
flags carry the most signal, so the class boundary lives in word choice and
context rather than length/punctuation statistics.

In [ ]:
display(Image(str(FIG / 'eda' / 'feature_correlation_heatmap.png')))
display(Image(str(FIG / 'eda' / 'length_by_class.png')))

## 3. Model comparison

Macro-F1 is the headline metric because of the class imbalance (a majority-only
classifier already scores well on accuracy).

In [ ]:
comp = pd.DataFrame([
    ['TF-IDF + Logistic Regression', base['accuracy'], base['macro_f1'], base['weighted_f1']],
    ['LogReg (grid-search tuned)', tune['after_test']['accuracy'], tune['after_test']['macro_f1'], tune['after_test']['weighted_f1']],
    ['DistilBERT (class-weighted)', dbert['accuracy'], dbert['macro_f1'], dbert['weighted_f1']],
], columns=['model', 'accuracy', 'macro_f1', 'weighted_f1']).set_index('model').round(3)
display(comp)
display(Image(str(FIG / 'model_comparison.png')))

## 4. Per-class results & error analysis (DistilBERT)

In [ ]:
pc = dbert['per_class']
percls = pd.DataFrame(
    [[k, pc[k]['precision'], pc[k]['recall'], pc[k]['f1-score'], int(pc[k]['support'])] for k in order],
    columns=['class', 'precision', 'recall', 'f1', 'support']).set_index('class').round(3)
display(percls)
display(Image(str(FIG / 'confusion_distilbert.png')))
display(Image(str(FIG / 'roc_distilbert.png')))

## 5. Hyperparameter tuning & feature importance

In [ ]:
print('best params:', tune['best_params'])
print(f"test macro-F1  {tune['before_test']['macro_f1']:.3f} -> {tune['after_test']['macro_f1']:.3f}")
display(Image(str(FIG / 'tuning_before_after.png')))
display(Image(str(FIG / 'feature_importance.png')))
display(Image(str(FIG / 'learning_curve.png')))

## 6. Ablation — do the models learn context or keywords?

Masking the modal trigger word drops accuracy but keeps both models above the
majority baseline, showing they learn sentence context, not a keyword lookup.

In [ ]:
ablation = pd.DataFrame([
    ['Logistic Regression', base['accuracy'], base_mask['accuracy'], base['macro_f1'], base_mask['macro_f1']],
    ['DistilBERT', dbert['accuracy'], dbert_mask['accuracy'], dbert['macro_f1'], dbert_mask['macro_f1']],
], columns=['model', 'acc_full', 'acc_masked', 'macroF1_full', 'macroF1_masked']).set_index('model').round(3)
display(ablation)
print('majority-class baseline accuracy:', round(max(stats.values())/total, 3))

## Takeaways

- DistilBERT leads on accuracy / weighted-F1; the tuned Logistic Regression leads on macro-F1.
- Both models learn context beyond the surface modal keyword (ablation).
- The dominant bottleneck is rare-class data volume, not model choice — the learning curve is still rising.

See `FINAL_REPORT.pdf` for the full write-up.